In [2]:
print("""
@File         : pd.dataframe.pivot_and_pd.pivot_table.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 13:58:16
@Email        : cuixuanstephen@gmail.com
@Description  : pd.DataFrame.pivot and pd.pivot_table
""")


@File         : pd.dataframe.pivot_and_pd.pivot_table.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 13:58:16
@Email        : cuixuanstephen@gmail.com
@Description  : pd.DataFrame.pivot and pd.pivot_table



In [3]:
import pandas as pd

我们已经看到 `pd.DataFrame.stack`、`pd.DataFrame.melt` 和 `pd.wide_to_long` 都可用于帮助将 pd.DataFrame 从宽格式转换为长格式。另一方面，我们只看到 `pd.Series.unstack` 帮助我们从长格式转换为宽格式，但该方法的缺点是要求我们在使用它之前分配适当的行索引。使用 `pd.DataFrame.pivot`，可以跳过任何中间步骤，直接从长格式转换为宽格式。

除了 `pd.DataFrame.pivot` 之外， pandas 还提供了 `pd.pivot_table` 函数，它不仅可以从长重塑为宽，还允许在重塑过程中执行聚合。

![Using pd.pivot_table to reshape with sum aggregation](../../IMAGES/FIG7-6.png)

In [4]:
df = pd.DataFrame([
    ["Texas", "apple", 12, 8],
    ["Arizona", "apple", 9, 10],
    ["Florida", "apple", 0, 6],
    ["Texas", "orange", 10, 4],
    ["Arizona", "orange", 7, 2],
    ["Florida", "orange", 14, 3],
    ["Texas", "banana", 40, 28],
    ["Arizona", "banana", 12, 17],
    ["Florida", "banana", 190, 42],
], columns=["state", "fruit", "number_grown", "number_eaten"])
df = df.convert_dtypes(dtype_backend="numpy_nullable")

df

,state,fruit,number_grown,number_eaten
0,Texas,apple,12,8
1,Arizona,apple,9,10
2,Florida,apple,0,6
3,Texas,orange,10,4
4,Arizona,orange,7,2
5,Florida,orange,14,3
6,Texas,banana,40,28
7,Arizona,banana,12,17
8,Florida,banana,190,42


In [8]:
df.set_index(['state', 'fruit']).unstack()

number_grown               number_eaten              
fruit          apple banana orange        apple banana orange
state                                                        
Arizona            9     12      7           10     17      2
Florida            0    190     14            6     42      3
Texas             12     40     10            8     28      4

`pd.DataFrame.pivot` 让我们通过一个方法调用来解决这个问题。此方法的基本用法需要 `index=` 和 `columns=` 参数，分别指示哪些列应出现在行和列索引中：

In [9]:
df.pivot(index=['state'], columns=['fruit'])

number_grown               number_eaten              
fruit          apple banana orange        apple banana orange
state                                                        
Arizona            9     12      7           10     17      2
Florida            0    190     14            6     42      3
Texas             12     40     10            8     28      4

`pd.DataFrame.pivot` 将获取未指定为 `index=` 或 `columns=` 参数的任何列，并尝试将该列转换为生成的 pd.DataFrame 的值。但是，如果不希望所有剩余列都成为透视 pd.DataFrame 的一部分，则可以使用 `values=` 参数指定要保留的内容。例如，如果我们只关心透视 `number_grown` 列并忽略 `number_eaten` 列，我们可以将其写成：

In [10]:
df.pivot(
    index=['state'], columns=['fruit'],
    values=['number_grown']
)

number_grown              
fruit          apple banana orange
state                             
Arizona            9     12      7
Florida            0    190     14
Texas             12     40     10

In [12]:
df.pivot(
    index=['state'], columns=['fruit'],
    values=['number_grown']
).droplevel(level=0, axis='columns')

fruit,apple,banana,orange
state,,,
Arizona,9,12,7
Florida,0,190,14
Texas,12,40,10


虽然 `pd.DataFrame.pivot` 对于重塑很有用，但它只能在用于构成行和列的值均不重复的情况下才有用。为了了解这一限制，让我们使用稍加修改的 pd.DataFrame 来展示不同州和年份的不同水果的消费或种植方式：

In [13]:
df = pd.DataFrame([
    ["Texas", "apple", 2023, 10, 6],
    ["Texas", "apple", 2024, 2, 8],
    ["Arizona", "apple", 2023, 3, 7],
    ["Arizona", "apple", 2024, 6, 3],
    ["Texas", "orange", 2023, 5, 2],
    ["Texas", "orange", 2024, 5, 2],
    ["Arizona", "orange", 2023, 7, 2],
], columns=["state", "fruit", "year", "number_grown", "number_eaten"])
df = df.convert_dtypes(dtype_backend="numpy_nullable")

df

,state,fruit,year,number_grown,number_eaten
0,Texas,apple,2023,10,6
1,Texas,apple,2024,2,8
2,Arizona,apple,2023,3,7
3,Arizona,apple,2024,6,3
4,Texas,orange,2023,5,2
5,Texas,orange,2024,5,2
6,Arizona,orange,2023,7,2


In [14]:
df.pivot(index=['state', 'year'],
         columns=['fruit'])

number_grown        number_eaten       
fruit               apple orange        apple orange
state   year                                        
Arizona 2023            3      7            7      2
        2024            6   <NA>            3   <NA>
Texas   2023           10      5            6      2
        2024            2      5            8      2

In [18]:
try:
    df.pivot(index='state', columns=['fruit'])
except ValueError as e:
    print("Just removing it from our pd.DataFrame.pivot arguments will raise an exception.")

Just removing it from our pd.DataFrame.pivot arguments will raise an exception.


In [20]:
pd.pivot_table(
    df, index=['state'],
    columns=['fruit'], values=['number_grown', 'number_eaten']
)

number_eaten        number_grown       
fruit          apple orange        apple orange
state                                          
Arizona          5.0    2.0          4.5    7.0
Texas            7.0    2.0          6.0    5.0

In [21]:
pd.pivot_table(
    df, index=['state'],
    columns=['fruit'], values=['number_grown', 'number_eaten'],
    aggfunc='sum'
)

number_eaten        number_grown       
fruit          apple orange        apple orange
state                                          
Arizona           10      2            9      7
Texas             14      4           12     10

对于更高级的用例，甚至可以为 `aggfunc=` 提供一个值字典，其中字典中的每个键/值对分别指定要应用的列和聚合类型：

In [23]:
pd.pivot_table(
    df, index='state', columns='fruit',
    aggfunc={'number_grown': ['sum', 'mean'],
             'number_eaten': ['min', 'max']}
)

number_eaten                     number_grown                    
                 max          min                mean          sum       
fruit          apple orange apple orange        apple orange apple orange
state                                                                    
Arizona            7      2     3      2          4.5    7.0     9      7
Texas              8      2     6      2          6.0    5.0    12     10